In [ ]:
import os
import sys
from PIL import Image
import torch
from torchvision import transforms
sys.path.append(os.path.abspath('../augmenter'))
from spectrogram_augmentation import CustomAugmenter

DATASET_PATH = '/home/ioannis/Desktop/spectrograms/dataset_v3_copy'
SPECTROGRAM_FOLDER = 'spectrograms'
KPI_FOLDER = 'KPIs'
OUTPUT_FOLDER = 'multi_channel_tensors'

#Select which KPI to add also
SELECTED_KPIS = ['Jitter', 'Latency', 'Noise', 'Packet_Loss_Count', 'SNR']

augmenter = CustomAugmenter()
to_tensor = transforms.ToTensor()

spectrogram_root = os.path.join(DATASET_PATH, SPECTROGRAM_FOLDER)
kpi_root = os.path.join(DATASET_PATH, KPI_FOLDER)
output_root = os.path.join(DATASET_PATH, OUTPUT_FOLDER)
os.makedirs(output_root, exist_ok=True)

# === PROCESS EACH SCENARIO ===
for scenario in os.listdir(spectrogram_root):
    scenario_spectrogram_path = os.path.join(spectrogram_root, scenario)
    if not os.path.isdir(scenario_spectrogram_path):
        continue

    print(f"\n[+] Processing scenario: {scenario}")

    scenario_kpi_path = os.path.join(kpi_root, scenario)
    scenario_output_path = os.path.join(output_root, scenario)
    os.makedirs(scenario_output_path, exist_ok=True)

    for fname in os.listdir(scenario_spectrogram_path):
        if not fname.endswith('.png'):
            continue

        spectro_img_path = os.path.join(scenario_spectrogram_path, fname)
        spectro_img = Image.open(spectro_img_path).convert('RGB')
        spectro_tensor = augmenter(spectro_img)

        kpi_tensors = []
        missing_kpi = False

        for kpi in SELECTED_KPIS:
            kpi_img_path = os.path.join(scenario_kpi_path, kpi, fname)
            if os.path.exists(kpi_img_path):
                kpi_img = Image.open(kpi_img_path).convert('L')
                kpi_tensor = to_tensor(kpi_img)
                kpi_tensors.append(kpi_tensor)
            else:
                print(f"[!] Missing KPI image: {kpi_img_path}")
                missing_kpi = True
                break

        if not missing_kpi:
            combined = torch.cat([spectro_tensor] + kpi_tensors, dim=0)
            out_file = os.path.join(scenario_output_path, fname.replace('.png', '.pt'))
            torch.save(combined, out_file)
        else:
            print(f"[-] Skipping {fname} in {scenario} due to missing KPI.")
